# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook guides you through loading and exploring the [FAIR² Mental Health Survey Dataset from Kilifi County, Kenya](https://sen.science/doi/10.71728/senscience.vcs2-05nj) using the `mlcroissant` library. All dataset entities (record sets, fields, columns) are referenced by their Croissant `@id` fields.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and instantiate the Croissant dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")
print(f"Version: {getattr(metadata, 'version', 'N/A')}")
print(f"License: {getattr(metadata, 'license', 'N/A')}")

## 2. Data Overview
Review the available record sets, fields, and their Croissant `@id` fields.

In [ ]:
# List record sets and their fields (`@id`) for inspection

def print_record_sets(ds):
    print('Available Record Sets:')
    for rs in ds.record_sets:
        print(f"- @id: {rs.id}, name: " + (rs.name if hasattr(rs, 'name') else ''))
        print("  Fields (by @id):")
        for field in rs.fields:
            print(f"    - {field.id} ({getattr(field, 'name', '')})")
        print()

print_record_sets(dataset)

## 3. Data Extraction
Extract and load records from a specific record set into a DataFrame. All Croissant references use their `@id`.

In [ ]:
# Choose the main record set @id -- update this if you want to work on a different record set.
# This dataset's main survey data record set typically has @id like 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/recordsets/survey' or similar.
# Let's auto-detect or set it manually after reviewing `print_record_sets` above.

# For example (replace by the actual @id printed above):
main_record_set_id = None
for rs in dataset.record_sets:
    if 'mental_health' in rs.id or 'survey' in rs.id or 'kilifi' in rs.id:
        main_record_set_id = rs.id
        break
if main_record_set_id is None:
    main_record_set_id = dataset.record_sets[0].id  # fallback to first recordset

record_sets_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}

for rs_id in record_sets_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df

print(f"Fields for record set: {main_record_set_id}")
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering, normalization, and grouping. Croissant fields are always referenced via their `@id` (dict column name in the DataFrame).

> **Note:** Replace `<numeric_field_id>` and `<group_field_id>` as relevant based on the DataFrame's columns above. For this dataset, plausible numeric fields include GAD-7, PHQ-9, or PSQ scores.

In [ ]:
# Example numeric field selection (update as per the DataFrame columns):

numeric_field_id = None
# Try to auto-select a main score field (edit this if needed)
for col in dataframes[main_record_set_id].columns:
    if 'gad7' in col.lower() or 'phq9' in col.lower() or 'psq' in col.lower():
        numeric_field_id = col
        break
if numeric_field_id is None:
    # Fallback: select the first numeric column
    for dt in dataframes[main_record_set_id].dtypes.iteritems():
        if pd.api.types.is_numeric_dtype(dt[1]):
            numeric_field_id = dt[0]
            break
if numeric_field_id is None:
    raise Exception("No numeric field found for filtering and normalization. Please edit this cell and set one explicitly (e.g., 'psq_score').")

# Filtering by a threshold (example: scores above 10)
threshold = 10
filtered_df = dataframes[main_record_set_id][dataframes[main_record_set_id][numeric_field_id] > threshold].copy()
print(f"Filtered records where {numeric_field_id} > {threshold} (count={len(filtered_df)}):")
print(filtered_df.head())

# Normalization (z-score)
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by a categorical field (e.g., gender or education). Update <group_field_id> if needed.
group_field_id = None
for col in dataframes[main_record_set_id].columns:
    if 'gender' in col.lower() or 'education' in col.lower():
        group_field_id = col
        break
if group_field_id is not None and group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
    print(f"Grouped normalized score by '{group_field_id}':")
    print(grouped_df[[numeric_field_id, f"{numeric_field_id}_normalized"]])
else:
    print("No suitable group field identified for grouping analysis.")

## 5. Visualization
Visualize the distributions or relationships between key fields in the main record set. 

*All plots reference columns by their Croissant `@id` as in the EDA step above.*

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the main numeric field
plt.figure(figsize=(7,4))
sns.histplot(dataframes[main_record_set_id][numeric_field_id], bins=20, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

# If group_field_id is available, plot grouped distributions
if group_field_id:
    plt.figure(figsize=(8,5))
    sns.boxplot(data=dataframes[main_record_set_id], x=group_field_id, y=numeric_field_id)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.show()

## 6. Conclusion
This notebook demonstrates end-to-end usage of the FAIR² Kilifi County Mental Health Survey dataset via the Croissant schema and `mlcroissant` library.

- The Croissant record sets, fields, and columns can be referenced in a reproducible, machine-readable way using their `@id`
- Exploratory steps show how to filter, normalize, and group data by demographic and mental health indicators
- Further analysis can use these structured references for trustworthy data science or ML-ready asset pipelines.

*Notebook generated for demonstration purposes. For your own analysis, review the field IDs and adjust the analysis steps as needed for your research questions.*